Clinical Readmission Prediction — Complete Machine Learning Pipeline

1. Import Libraries

In [ ]:
pip install scikit-learn --default-timeout=100

In [6]:
# Core Libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

# Explainability
from sklearn.inspection import permutation_importance
import shap

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'sklearn'

2. Load Dataset

In [ ]:
# Load dataset
file_path = 'hospital_readmission_dataset.csv'
df = pd.read_csv(file_path)

# Display first rows
df.head()

3. Basic Dataset Information

In [ ]:
print('Dataset Shape:', df.shape)
print('\nColumns:\n', df.columns)
print('\nData Types:\n')
print(df.dtypes)

4. Missing Values

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)

5. Statistical Summary

In [ ]:
print(df.describe())

6. Target Variable Distribution

In [ ]:
plt.figure(figsize=(6,4))
df['label'].value_counts().plot(kind='bar')
plt.title('Target Variable Distribution')
plt.xlabel('Readmission Label')
plt.ylabel('Count')
plt.show()

7. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12,8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

8. Age Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['age'], bins=30, kde=True)
plt.title('Age Distribution')
plt.show()

9. Readmission by Gender

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='gender', hue='label')
plt.title('Readmission by Gender')
plt.show()

10. Readmission by Diagnosis

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='primary_diagnosis', hue='label')
plt.xticks(rotation=45)
plt.title('Readmission by Diagnosis')
plt.show()

11. Feature Engineering

In [ ]:
# Convert date column

df['admission_date'] = pd.to_datetime(df['admission_date'])

# Extract date features

df['admission_year'] = df['admission_date'].dt.year
df['admission_month'] = df['admission_date'].dt.month

# Drop unnecessary columns

df.drop(columns=['patient_id', 'admission_date'], inplace=True)

print(df.head())

12. Define Features and Target

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

13. Identify Numerical and Categorical Columns

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(exclude=['object']).columns

print('Categorical Columns:', categorical_cols)
print('\nNumerical Columns:', numerical_cols)

14. Preprocessing Pipeline

In [ ]:
# Numerical pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

15. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train Shape:', X_train.shape)
print('Test Shape:', X_test.shape)

In [10]:
# Read Excel file
df = pd.read_csv("hospital_readmission_dataset.csv")

# Show first 5 rows
df.head()

,patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
0,P00001,2022-04-14,Spring,66,Male,South,Diabetes,5,6,Interventional,8,6,1,Medicare,Home Health,0.92,1
1,P00002,2021-09-19,Fall,55,Male,South,Diabetes,4,6,Interventional,6,4,3,Private,Home Health,0.88,1
2,P00003,2023-04-12,Spring,69,Female,West,Hypertension,6,8,Medical,9,6,2,Medicare,Skilled Nursing,0.97,1
3,P00004,2023-08-14,Summer,83,Male,South,Stroke,6,11,Medical,11,4,2,Medicare,Skilled Nursing,0.97,1
4,P00005,2021-11-05,Fall,54,Female,North,Stroke,4,10,Medical,6,2,1,Uninsured,Home Health,0.83,1
